In [1]:
#! pip uninstall typing-extensions --yes
#! pip install typing-extensions==4.14.0 

Found existing installation: typing_extensions 4.7.1
Uninstalling typing_extensions-4.7.1:
  Successfully uninstalled typing_extensions-4.7.1
     ---------------------------------------- 43.8/43.8 kB 1.1 MB/s eta 0:00:00


In [113]:
import pandas as pd
import numpy as np
from datetime import datetime,timedelta
from lxml import html

import requests
from bs4 import BeautifulSoup
import selenium
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.common.keys import Keys
#!pip install requests_html
#from requests_html import HTMLSession
import random
import re
import time as  t


import string
import matplotlib as mlt
import matplotlib.pyplot as plt
%matplotlib inline

from sklearn.preprocessing import LabelEncoder

import pymysql
pymysql.install_as_MySQLdb()
import MySQLdb

#! pip install wordcloud
#from subprocess import check_output
#from wordcloud import WordCloud, STOPWORDS

In [115]:
def merge(dict1, dict2):
    return(dict2.update(dict1))

# Extract HTML - Paginated Func
def extract(result_start):    
    url = f"https://www.linkedin.com/jobs-guest/jobs/api/seeMoreJobPostings/search?f_TPR=r604800&geoId=103644278&keywords=Sport&location=United%20States&start={result_start}"
    
    user_agents_list = [
    'Mozilla/5.0 (iPad; CPU OS 12_2 like Mac OS X) AppleWebKit/605.1.15 (KHTML, like Gecko) Mobile/15E148',
    'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/99.0.4844.83 Safari/537.36',
    'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0 Safari/537.36',
    'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/99.0.4844.51 Safari/537.36',
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/144.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/145.0.0.0 Safari/537.36"]
    
    headers = {"accept":"*/*",
               "accept-encoding": "gzip, deflate, br, zstd",
               "accept-language": "en-US,en;q=0.9",
               'User-Agent': random.choice(user_agents_list)}

    page = requests.get(url,headers = headers)

    soup = BeautifulSoup(page.content,'html.parser')
    return(soup)

def extract_inner(url_inner):
       
    user_agents_list = [
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/145.0.0.0 Safari/537.36"]
    
    
    headers = {"accept":"*/*",
               "accept-encoding": "gzip, deflate, br, zstd",
               "accept-language": "en-US,en;q=0.9",
               'User-Agent': random.choice(user_agents_list)}

    r_inner = requests.get(url_inner,headers)
    soup_inner = BeautifulSoup(r_inner.content,'html.parser')
    return(soup_inner)

def transform(soup):
    divs = soup.find_all('li')
    
    for job in divs:
        raw_job={         
                'title': job.find('h3',class_='base-search-card__title').get_text().strip() if job.find('h3',class_='base-search-card__title') else None,
                'company': job.find('h4',class_ = 'base-search-card__subtitle').get_text().strip() if job.find('h4',class_ = 'base-search-card__subtitle') else None,
                'location_info': job.find('span',class_ = 'job-search-card__location').get_text().strip() if job.find('span',class_ = 'job-search-card__location') else None,
                'posting_date': job.find('time').get_text().strip() if job.find('time') else None
                }
        raw_job['job_url'] = job.a['href'].partition('?position')[0] if job.a['href'] else None 
        raw_job['job_id'] = raw_job['job_url'].split('-')[-1]
    
        raw_job['scrape_datetime'] = datetime.now()         
        
        #details = []
        more_info = extract_inner(raw_job['job_url'])
        raw_job['details'] = detail_fetcher(more_info)
        
        jobs_list.append(raw_job)

    return


In [116]:
# For final intergration for chunk in range(0, 241,10):
jobs_list = []
i = 1
for chunk in range(0, 11,10):
    recent_jobs_html = extract(chunk)
    transform(recent_jobs_html)
    t.sleep(4)# adding this due to a limit on the scrape frequency
    print(f'Through {i} iteration(s) we have yielded {len(jobs_list)} results')
    i+=1

Through 1 iteration(s) we have yielded 10 results
Through 2 iteration(s) we have yielded 20 results


In [38]:
joblist = []
c = extract()

https://www.linkedin.com/jobs-guest/jobs/api/seeMoreJobPostings/search?f_TPR=r604800&geoId=103644278&keywords=Sport&location=United%20States&start=0


In [70]:
transform(c)

{'title': 'Manager, Athlete Marketing', 'company': 'Red Bull', 'location_info': 'Santa Monica, CA', 'posting_date': '3 days ago', 'job_url': 'https://www.linkedin.com/jobs/view/manager-athlete-marketing-at-red-bull-4371481211', 'job_id': '4371481211'}
{'title': '1000 - Recreation Specialist', 'company': 'City of Fort Walton Beach', 'location_info': 'Fort Walton Beach, FL', 'posting_date': '5 days ago', 'job_url': 'https://www.linkedin.com/jobs/view/1000-recreation-specialist-at-city-of-fort-walton-beach-4401451818', 'job_id': '4401451818'}
{'title': 'Partnership Activation Coordinator', 'company': 'Detroit City FC', 'location_info': 'Detroit, MI', 'posting_date': '3 days ago', 'job_url': 'https://www.linkedin.com/jobs/view/partnership-activation-coordinator-at-detroit-city-fc-4400867347', 'job_id': '4400867347'}
{'title': 'Sport Scientist', 'company': 'Angel City Football Club', 'location_info': 'Thousand Oaks, CA', 'posting_date': '6 days ago', 'job_url': 'https://www.linkedin.com/job

In [111]:
def detail_fetcher(temp_text):
    try:
        job_inner = temp_text.find('div', class_ = "show-more-less-html__markup show-more-less-html__markup--clamp-after-5 relative overflow-hidden")
        #active_job_tag = (False if job_soup.find('div', class_ = "opportunity-preview__closed") else True)
        job_details = job_inner.find_all('strong')
        
        data = {}
        headers = []
        for section in job_details:
            if not section:
                continue
            
            details = []
                
            header_text = section.get_text()
            next_tag = section.find_next_sibling()
        
            if (next_tag and next_tag.name in ['ul', 'ol']):
                headers.append(header_text)
                items = next_tag.find_all('li')
                details = [item.get_text(strip=True) for item in items]
            
                data[header_text] = details 
                    
        
        # Convert to single-row DataFrame
        details_df = pd.DataFrame({k: pd.Series(v) for k, v in data.items()})
    except:
        details_df = []

    return(details_df)

In [104]:
print(temp_text)

<!DOCTYPE html>

<html lang="en">
<head>
<meta content="d_jobs_guest_details" name="pageKey"/>
<meta content="max-image-preview:large, noarchive" name="robots"/>
<meta content="max-image-preview:large, archive" name="bingbot"/>
<!-- --><!-- --> <meta content="en_US" name="locale"/>
<!-- --> <meta data-app-version="2.0.2901" data-browser-id="250e364d-04fa-4de7-8b03-3266386499d7" data-call-tree-id="AAZP7gLNcp7qrfV7bJG0MA==" data-dfp-member-lix-treatment="control" data-disable-jsbeacon-pagekey-suffix="false" data-dna-member-lix-treatment="enabled" data-enable-page-view-heartbeat-tracking="" data-human-member-lix-treatment="enabled" data-is-bot="true" data-is-epd-audit-event-enabled="false" data-is-feed-sponsored-tracking-kill-switch-enabled="false" data-member-id="0" data-multiproduct-name="jobs-guest-frontend" data-network-interceptor-lix-value="control" data-page-instance="urn:li:page:d_jobs_guest_details;qlSXl9mpSx+CAtNQR2gOJw==" data-recaptcha-v3-integration-lix-value="control" data-s

In [103]:
temp_text = extract_inner('https://www.linkedin.com/jobs/view/manager-athlete-marketing-at-red-bull-4371481211/')
detail_fetcher(temp_text)

All the responsibilities we'll trust you with:
<strong>ATHLETE MARKETING<br/><br/></strong>
ATHLETE MARKETING
<br/>
ATHLETE PERFORMANCE MANAGEMENT
<br/>
DAY TO DAY ADMINISTRATION
<br/>
Your areas of knowledge and expertise that matter most for this role:
<ul><li>Minimum of 3–5 years of experience in sports marketing</li><li>In-depth knowledge of sports management and the sports industry</li><li>Exceptional communication skills, including the ability to train and deliver presentations</li><li>Demonstrated success in planning, organization, and project management</li><li>Strong strategic and creative thinking abilities</li><li>Openness to new environments, ideas, processes, innovations, and possibilities</li><li>Entrepreneurial mindset; an autonomous self-starter focused on results and consumer needs</li><li>Employees may be required to lift and or move up to 25 pounds or more as needed<br/><br/></li></ul>


,Your areas of knowledge and expertise that matter most for this role:
0,Minimum of 3–5 years of experience in sports m...
1,In-depth knowledge of sports management and th...
2,"Exceptional communication skills, including th..."
3,"Demonstrated success in planning, organization..."
4,Strong strategic and creative thinking abilities
5,"Openness to new environments, ideas, processes..."
6,Entrepreneurial mindset; an autonomous self-st...
7,Employees may be required to lift and or move ...


In [106]:
print(temp_text.find('span', class_ = "description__job-criteria-text description__job-criteria-text--criteria").get_text().strip())
print(temp_text.find('div', class_ = "show-more-less-html__markup show-more-less-html__markup--clamp-after-5 relative overflow-hidden"))

Mid-Senior level
<div class="show-more-less-html__markup show-more-less-html__markup--clamp-after-5 relative overflow-hidden">
          Red Bull gives wings to people and ideas. In this role, the Athlete Marketing Manager (AMM) helps athletes advance in their sport and achieve their goals. The AMM is responsible for bringing Red Bull’s global brand to life at the regional level through athletes, athlete-led projects, and athlete marketing initiatives.<br/><br/>The Athlete Marketing Manager will develop and manage Red Bull’s presence by using branded athletes and product-placement athletes as key assets to expand the consumer base and create clear points of brand differentiation. In addition, the Athlete Marketing Manager must have the strategic vision and practical execution skills to extend beyond the regional level and leverage managed athletes on a national and global scale.<br/><br/><strong>All the responsibilities we'll trust you with:<br/><br/></strong><strong>ATHLETE MARKETING<

In [117]:
temp_df = pd.DataFrame(jobs_list)

In [129]:
pd.set_option('display.max_colwidth',None)
temp_df

title  \
0                            Manager, Athlete Marketing   
1                          1000 - Recreation Specialist   
2                    Partnership Activation Coordinator   
3                                       Sport Scientist   
4                               PT Operations Assistant   
5                                           CRM Analyst   
6                   Assistant Athletic Director (26-27)   
7              Digital Marketing Manager, Email and Web   
8                       Coordinator, Stadium Operations   
9   Sports Research Athlete Insights Intern (12 Months)   
10                       Rams Ambassador (Gameday role)   
11                         Manager, Marketing Solutions   
12                                       AP Coordinator   
13                       Director - Football Operations   
14                             Sports Management Intern   
15                            Soccer Operations Manager   
16  Assistant Director of Women's Basketball Operations   
17                       Vice President Of Global Sales   
18                             DIR BASKETBALL OPER-MENS   
19                    Baseball Sport Coordinator Makiki   

                         company          location_info posting_date  \
0                       Red Bull       Santa Monica, CA   3 days ago   
1      City of Fort Walton Beach  Fort Walton Beach, FL   5 days ago   
2                Detroit City FC            Detroit, MI   3 days ago   
3       Angel City Football Club      Thousand Oaks, CA   6 days ago   
4   Houston Dynamo Football Club            Houston, TX   3 days ago   
5   Houston Dynamo Football Club            Houston, TX   6 days ago   
6   Poplar Bluff School District       Poplar Bluff, MO    1 day ago   
7   Houston Dynamo Football Club            Houston, TX   2 days ago   
8                 Inter Miami CF              Miami, FL    1 day ago   
9                    New Balance             Boston, MA   2 days ago   
10              Los Angeles Rams          Inglewood, CA   6 days ago   
11                           WWE           New York, NY   3 days ago   
12  Houston Dynamo Football Club            Houston, TX  5 hours ago   
13     Atlantic Coast Conference          Charlotte, NC    1 day ago   
14                     i9 Sports     Salt Lake City, UT   1 hour ago   
15              Tampa Bay Sun FC              Tampa, FL   6 days ago   
16        Stony Brook University        Stony Brook, NY   4 days ago   
17                     GAT Sport    Fort Lauderdale, FL   4 days ago   
18     Loyola University Chicago            Chicago, IL   4 days ago   
19                     i9 Sports           Honolulu, HI   1 hour ago   

                                                                                                                        job_url  \
0                                           https://www.linkedin.com/jobs/view/manager-athlete-marketing-at-red-bull-4371481211   
1                         https://www.linkedin.com/jobs/view/1000-recreation-specialist-at-city-of-fort-walton-beach-4401451818   
2                           https://www.linkedin.com/jobs/view/partnership-activation-coordinator-at-detroit-city-fc-4400867347   
3                                     https://www.linkedin.com/jobs/view/sport-scientist-at-angel-city-football-club-4402060956   
4                         https://www.linkedin.com/jobs/view/pt-operations-assistant-at-houston-dynamo-football-club-4400857709   
5                                     https://www.linkedin.com/jobs/view/crm-analyst-at-houston-dynamo-football-club-4399577912   
6               https://www.linkedin.com/jobs/view/assistant-athletic-director-26-27-at-poplar-bluff-school-district-4404300411   
7         https://www.linkedin.com/jobs/view/digital-marketing-manager-email-and-web-at-houston-dynamo-football-club-4400875624   
8                                https://www.linkedin.com/jobs/view/coordinator-stadium-operations-at-inter-miami-cf-4

In [ ]:
joblist = []
leagues = ['Major%20League%20Soccer', 'Major%20League%20Baseball','National%20Football%20League']
for league in leagues:
    c=extract(league)
    transform(c)
    
joblist1 = joblist

In [ ]:
joblist = []
leagues_again = ['National%20Hockey%20League', 'National%20Basketball%20Association']
for league in leagues_again:
    c=extract(league)
    transform(c)
    
joblist2 = joblist

In [ ]:
database = MySQLdb.connect(host="localhost" , user="root" , passwd="Pps11844")
cursor = database.cursor()

def execute_query(query_statement):
    try:
        cursor.execute(query_statement);
        database.commit();
        print("Data is Succefully Inserted")
    
    except Exception as e :
        database.rollback();
        print("The  Exception Occured : ", e)

execute_query("USE JobsinSports")

SQL_df_posting = pd.read_sql('select * from job_posting',database)

SQL_df_companies = pd.read_sql('select * from company_team',database)

cursor.execute("SELECT MAX(company_ID) FROM company_team;")
result = cursor.fetchone();
max_comp_ID = result[0]

database.close()

In [ ]:
job_posting_linkedin_1 = pd.DataFrame(joblist1)
job_posting_linkedin_2 = pd.DataFrame(joblist2)
job_posting_linkedin = pd.concat([job_posting_linkedin_1, job_posting_linkedin_2])
job_posting_linkedin.reset_index(drop=True, inplace=True)

job_posting_linkedin['job_ID'] = job_posting_linkedin['job_ID'].astype(float)

for i,j in job_posting_linkedin.iterrows():
    if(re.findall(r'\$',j['additionals'])):
        job_posting_linkedin.at[i,'salary'] = '$'+(j['additionals'].partition('$')[2].partition('.')[0].partition('<')[0].partition('/')[0].partition('\\')[0].replace(' ','').replace('to','-').replace('TO','-').replace('K',',000').replace('k',',000').replace('\n',',000'))
    else:
        job_posting_linkedin.at[i,'salary'] = 'NA'

for i,j in job_posting_linkedin.iterrows():
    if(len(j['salary'])==3):
        job_posting_linkedin.at[i,'salary'] = (j['salary'] + '/hr')
    else:
        pass

for i,j in job_posting_linkedin.iterrows():
    if(re.findall(r'New York City',j['Location'])):
        job_posting_linkedin.at[i,'Location'] = 'New York, NY'
    else:
        pass
    
job_posting_linkedin['job_city'] = job_posting_linkedin['Location'].str.partition(",")[0]
job_posting_linkedin['job_state'] = job_posting_linkedin['Location'].str.partition(",")[2]

job_posting_linkedin['Company'] = job_posting_linkedin['Company'].str.partition('(')[0].str.replace('Football Club ','FC').str.replace('Football Club','FC').str.replace('Soccer Club','SC')
job_posting_linkedin['Company'] = job_posting_linkedin['Company'].str.strip()
job_posting_linkedin['posting_source_ID'] = 3
job_posting_linkedin['application_deadline'] = 'Unknown'
job_posting_linkedin['scrape_datetime'] = pd.to_datetime(job_posting_linkedin['scrape_datetime'])
job_posting_linkedin['posting_datetime'] = pd.to_datetime(job_posting_linkedin['posting_datetime'])


job_requirements_df = pd.DataFrame(job_posting_linkedin[['job_ID','details']])
job_requirements_df_final = job_requirements_df.assign(temp = job_requirements_df.details.str.split(",")).explode('details').drop('temp',axis=1)
job_requirements_df_final['details'] = job_requirements_df_final['details'].str.replace("'","").str.replace('"','')

In [ ]:
job_posting_linkedin.drop(['Location','details','additionals'],axis = 1,inplace = True)

In [ ]:
Company_Team = pd.DataFrame(job_posting_linkedin['Company'])
Company_Team_df = Company_Team.drop_duplicates()

In [ ]:
Company_Team_df['Company_temp'] = [1,2,3,4,5,6,7,8]
Company_Team_df.loc[Company_Team_df['Company_temp'] == 1,'company_ID'] = int(max_comp_ID + 1)
Company_Team_df.loc[Company_Team_df['Company_temp'] == 5,'company_ID'] = int(max_comp_ID + 2)
Company_Team_df.loc[Company_Team_df['Company_temp'] == 6,'company_ID'] = int(max_comp_ID + 3)
Company_Team_df.loc[Company_Team_df['Company_temp'] == 7,'company_ID'] = int(max_comp_ID + 4)
Company_Team_df.loc[Company_Team_df['Company_temp'] == 8,'company_ID'] = int(max_comp_ID + 5)
Company_Team_df.loc[Company_Team_df['Company_temp'] == 2,'company_ID'] = 258
Company_Team_df.loc[Company_Team_df['Company_temp'] == 3,'company_ID'] = 257
Company_Team_df.loc[Company_Team_df['Company_temp'] == 4,'company_ID'] = 255
Company_Team_df.drop('Company_temp',inplace=True,axis=1)

In [ ]:
Company_Team_df

In [ ]:
job_posting_linkedin_df = pd.merge(job_posting_linkedin, Company_Team_df, left_on="Company", right_on="Company", how='left')

In [ ]:
job_posting_linkedin_df = job_posting_linkedin_df.reindex(columns = ['job_ID','title',"company_ID",'posting_source_ID','posting_datetime','scrape_datetime','application_deadline', 'salary','job_city','job_state','url'])

In [ ]:
Sources = pd.DataFrame({'source_ID': [3], 'source_name': ['Linkedin']})

In [ ]:
# Tested but not perfected yet -- IGNORE
#job_posting_linkedin_df
#count = 1

#for i,j in Company_Team_df.iterrows():
#  print(i, j['Company'])
#    if (j['Company'] in SQL_df_companies['company_name'].values):
#        j.at[i,'company_ID'] = SQL_df_companies['company_ID']
#    else:
#        Company_Team_df.at[i,'company_ID'] = max_comp_ID + count
#        count = count + 1


In [ ]:
## Initialize connection to MYSQL
database = MySQLdb.connect(host="localhost" , user="root" , passwd="Pps11844")
cursor = database.cursor()

In [ ]:
def execute_query(query_statement):
    try:
        cursor.execute(query_statement);
        database.commit();
        print("Data is Succefully Inserted")
    
    except Exception as e :
        database.rollback();
        print("The  Exception Occured : ", e)


In [ ]:
execute_query("USE JobsinSports")


In [ ]:
for i,j in job_requirements_df_final.iterrows():
    execute_query('INSERT INTO Job_Requirements (job_ID, requirements) VALUES (%d, "%s")' % (j['job_ID'],j['details']))

In [ ]:
for i,j in Sources.iterrows():
    execute_query('INSERT INTO Sources (source_ID, source_name) VALUES (%d, "%s")' % (j['source_ID'],j['source_name']))

In [ ]:
for i,j in Company_Team_df.iterrows():
    execute_query('INSERT INTO Company_Team (company_ID, company_name) VALUES (%d, "%s")' % (j['company_ID'], j['Company']))

In [ ]:
for i,j in job_posting_linkedin_df.iterrows():
    execute_query('INSERT INTO Job_Posting (job_ID, job_title, company_ID, posting_datetime, scraped_datetime, salary, job_city, job_state, posting_url, source_identifier) VALUES (%d, "%s", %d, "%s","%s", "%s", "%s", "%s", "%s", %d)' % (j['job_ID'], j['title'], j['company_ID'], j['posting_datetime'], j['scrape_datetime'], j['salary'], j['job_city'], j['job_state'], j['url'], j['posting_source_ID']))

In [ ]:
database.close()

THIS IS UNUSED BUT INTERESTING CODE ID LIKE TO RETAIN.

I ran into issues becacuse instead of having paginated data the data was on an infinite scroll. To address this I 
implemented Selenium but ran into issues with the scrape as a user auth was needed. I went around this with an automated
escape action but the issue persisted as I wasnt getting any scroll. I was able to accheive a scroll by increasing scroll size 
and wait time to allow buffer.

The problem was I would get caught in an infinte buffer which yielded fewer results than I hoped for. I was able to find the succesful request which brought me to the Linkedin API which i used instead with the code herein. 

In [13]:
def extract():
    url = 'https://www.linkedin.com/jobs/search/?f_TPR=r604800&geoId=103644278&keywords=Sport&location=United%20States '
    chrome_options = Options()
    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("disable-infobars")
    chrome_options.add_argument("--disable-extensions")
    chrome_options.add_argument("--disable-dev-shm-usage")
    
    driver = webdriver.Chrome()
    driver.maximize_window()
    
    driver.get(url)
    
    actions = ActionChains(driver)
    actions.send_keys(Keys.ESCAPE).perform()

    last_height = 0
    unchanged_count = 0

    while True:
        # Scroll to the bottom
        driver.execute_script("window.scrollBy(0,5000)")
        time.sleep(5)  # Wait for new jobs to load

        new_height = driver.execute_script("return document.body.scrollHeight")
        print(str(new_height)+"-"+str(last_height))

        if new_height == last_height:
            unchanged_count += 1
            print(f"No new content ({unchanged_count}/3)")
            if unchanged_count >= 3:
                print("Reached end of page.")
                break
        else:
            unchanged_count = 0
            last_height = new_height

    # Once fully scrolled, hand the full HTML to BeautifulSoup
    soup = BeautifulSoup(driver.page_source, 'html.parser')
    return soup

In [19]:
def extract():
    url = 'https://www.linkedin.com/jobs/search/?f_TPR=r604800&geoId=103644278&keywords=Sport&location=United%20States'
    chrome_options = Options()
    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("disable-infobars")
    chrome_options.add_argument("--disable-extensions")
    chrome_options.add_argument("--disable-dev-shm-usage")
    
    driver = webdriver.Chrome()
    driver.maximize_window()
    
    driver.get(url)
    time.sleep(5)  # Wait for initial page load
    
    actions = ActionChains(driver)
    actions.send_keys(Keys.ESCAPE).perform()
    
    last_height = driver.execute_script("return document.body.scrollHeight")  # Get initial height
    unchanged_count = 0
    current_position = 0

    while True:
        # --- Scroll down gradually in increments ---
        while current_position < last_height:
            current_position += 800  # Scroll 600px at a time
            driver.execute_script(f"window.scrollTo(0, {current_position});")
            time.sleep(0.8)  # Small pause between each step to let content trigger

        # --- Now at the bottom, wait for buffer to finish loading ---
        print("Waiting for buffer...")
        time.sleep(5)

        new_height = driver.execute_script("return document.body.scrollHeight")
        print(str(new_height) + "-" + str(last_height))

        if new_height == last_height:
            unchanged_count += 1
            print(f"No new content ({unchanged_count}/3)")
            if unchanged_count >= 3:
                print("Reached end of page.")
                break
        else:
            unchanged_count = 0
            last_height = new_height

    # Once fully scrolled, hand the full HTML to BeautifulSoup
    soup = BeautifulSoup(driver.page_source, 'html.parser')
    return soup